In [1]:
# Import libraries

from openai import OpenAI
import torch as t
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import PyPDF2
import pprint

from IPython.display import display, Markdown
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


c:\Users\moosa\OneDrive\Documents\windows_dev\tripos_bot\openai-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
"""
Helper functions for text extraction, embedding, context retrieval for RAG.
"""

def extract_text_from_pdf(pdf_path):
    """
    Given a file path to a PDF, return the text as a string.
    """
    with open(pdf_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        text = ''
        for page_num in range(len(reader.pages)):
            text += reader.pages[page_num].extract_text()
    return text

def clean_and_chunk_text(text):
    """
    Given a string of text, clean it by doing the following:
    - Remove whitespace
    - Remove some special characters
    Return the text as a list of paragraphs (list of strings).
    """
    cleaned_text = re.sub(r'\s+', ' ', text)
    pattern = re.compile(r'(\(\w\))')
    chunks = pattern.split(cleaned_text)
    merged_chunks = []
    for i in range(1, len(chunks), 2):
        merged_chunks.append(f'{chunks[i]} {chunks[i+1].strip()}')
    return merged_chunks

def get_embeddings(chunks, model):
    """
    Given a list of strings (chunks), return the chunk embeddings.
    """
    return model.encode(chunks)

def retrieve_context(query_embedding, chunk_embeddings, chunks, N):
    """
    Given a query embedding, and chunk embeddings, return the N most relevant chunks to the query.
    """
    similarities = cosine_similarity([query_embedding], chunk_embeddings)
    sorted_indices = np.argsort(similarities[0])[::-1]
    relevant_chunks = [chunks[i] for i in sorted_indices[:N]]
    return relevant_chunks

def convert_latex_format(math_string):
    # Replace inline math from \(...\) to $...$
    inline_converted = re.sub(r'\\\((.*?)\\\)', r'$\1$', math_string)
    
    # Replace display math from \[...\] to $$...$$
    display_converted = re.sub(r'\\\[([\s\S]*?)\\\]', r'$$\1$$', inline_converted)
    
    return display_converted

In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [4]:
# Extract text from user PDF and convert to chunks after cleaning
test_paper = extract_text_from_pdf("IA 1P1 2018.pdf")
test_paper_chunks = clean_and_chunk_text(test_paper)

In [5]:
# User query and query embedding
query = "Please generate a very difficult exam problem on angular momentum with multiple parts that might appear in a university engineering exam."
query_embedding = get_embeddings([query], model)[0]

In [6]:
# Generate embeddings
test_paper_chunks_embeddings = get_embeddings(test_paper_chunks, model)

In [7]:
context = retrieve_context(
    query_embedding=query_embedding,
    chunk_embeddings=test_paper_chunks_embeddings,
    chunks=test_paper_chunks,
    N=5,
)
context = "\n".join(context)
pprint.pprint(context)

('(a) Obtain expressions and values for the rad ial and polar components of '
 'the acceleration of the satellite in the elliptical orbit. Explain why the '
 'angular momentum of the satellite is conserved about an axis passing through '
 'the centre of the earth. [6]\n'
 '(a) the angular acceleration of the rod ; [5]\n'
 '(b) Find the angular frequency, \uf0771, of the normal mode in which the '
 'mass moves up and dow n in the z direction. [5]\n'
 '(b) The angul ar velocity of the spinner about the central axis is increased '
 'linearly as a function of time t such that ct=\uf071\uf026 , where c is a '
 'constant. Find the total force F\n'
 '(a) Find the moment of inertia of the fidget spinner about an axis through '
 'its centre and perpendicular to the plane of the masses. [6]')


In [8]:
AUGMENTED_INPUT = f"{query}. Style of questions to draw inspiration from: {context}"

In [13]:
# System prompt for question generator (unrefined)
UNREFINED_QUESTON_SYSPROMPT = """
You are an expert question setter for university level exams for maths, engineering, and physics.

Your job is to produce difficult exam problems at a university level on demand when prompted by the user.

The questions should be creative and test the student's understanding thoroughly.

The question must be solveable, and logically consistent.

You must only return the exam problem. Do not provide the solution or any other information as this might allow the student to cheat.

Return your response in Markdown.
"""

In [9]:
# System prompt for question refiner, given unrefined question
REFINED_QUESTION_SYSPROMPT = """
You are a quality control officer for university engineering exam problems.

You must only do the following tasks:
- IF refinements are needed, return the modified question with the improvements made.
- IF no refinements are needed, simply return the original question.

Return your response in Markdown.
"""

In [10]:
# System prompt for answering refined question (unrefined)
UNREFINED_ANSWER_SYSPROMPT = """
You are a specialized AI agent who is an expert at answering university level exam problems in physics, maths, and engineering.

Given a university exam problem(s), you must provide detailed solutions to the problem(s).

The solutions must be clear, logical, and correct.

Return your response in Markdown.
"""

In [11]:
# Create OpenAI client for sending API requests
client = OpenAI()

In [14]:
# Produced unrefined question
unrefined_question_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role" : "system", "content" : UNREFINED_QUESTON_SYSPROMPT},
        {"role" : "user", "content" : AUGMENTED_INPUT}
    ],
    temperature=0.5,
)
unrefined_question = unrefined_question_response.choices[0].message.content

In [19]:
display(Markdown(convert_latex_format(unrefined_question)))

**University Engineering Exam Problem: Angular Momentum**

Consider a system of three particles of masses $ m_1, m_2, $ and $ m_3 $ respectively, moving in three-dimensional space. The particles are located at positions $ \mathbf{r}_1 = x_1 \mathbf{i} + y_1 \mathbf{j} + z_1 \mathbf{k}, $ $ \mathbf{r}_2 = x_2 \mathbf{i} + y_2 \mathbf{j} + z_2 \mathbf{k}, $ and $ \mathbf{r}_3 = x_3 \mathbf{i} + y_3 \mathbf{j} + z_3 \mathbf{k}. $

(a) Calculate the total angular momentum of the system with respect to the origin at $ O(0,0,0) $ in vector form.

(b) If the particles interact through central forces only, prove that the angular momentum of the system is conserved.

(c) Given that the particles are subject to an external torque $ \mathbf{\tau} = \tau_0 \sin(\omega t) \mathbf{i}, $ determine the equations of motion for the system and discuss the behavior of the system in the long run.

(d) Suppose that the system is constrained to move on a sphere of radius $ R $ centered at the origin, derive the expression for the angular velocity of the system in terms of the Lagrange multipliers.

(e) Finally, if the system is perturbed slightly from its equilibrium configuration, discuss the small oscillations about this equilibrium and determine the frequencies of oscillation for the system.

[Note: You may assume appropriate physical laws and relationships to solve the problem.]

In [16]:
# Modify the unrefined_question prompt slightly
REFINED_QUESTION_INSTRUCTIONS = f"""
Refine the following exam problems to ensure they are clear and suitable for a first year university engineering exam:

{unrefined_question}
"""

In [17]:
# Feed unrefined question into model to get refined questions
refined_question_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role" : "system", "content" : REFINED_QUESTION_SYSPROMPT},
        {"role" : "user", "content" : REFINED_QUESTION_INSTRUCTIONS},
    ]
)
refined_question = refined_question_response.choices[0].message.content

In [18]:
print(refined_question)

This question is clear and suitable for a first-year university engineering exam. 

RETURN: 
A steam power plant operates on the Rankine cycle with superheat and reheat. Steam enters the first-stage turbine at 8 MPa and 500°C, expands to 2 MPa, where it is reheated to 450°C, then expands in the second-stage turbine to the condenser pressure of 20 kPa. The isentropic efficiencies of the turbines are 90%. The pump and all piping are isentropic. Determine the thermal efficiency of the cycle. Additionally, calculate the specific work output of the turbines per unit mass of steam flowing, the heat transfer for the two stages of heat addition, and the mass flow rate of steam in kg/s if the net power output is 50 MW. Given: Specific heat of superheated steam at constant pressure = 2.1 kJ/kg·K, Specific heat of superheated steam at constant volume = 1.7 kJ/kg·K, Specific heat of saturated liquid water = 4.2 kJ/kg·K, Specific heat of saturated vapor water = 2.1 kJ/kg·K, Latent heat of vaporizat

In [19]:
# Create a user prompt for passing in the refined question to the model with instructions
UNREFINED_ANSWER_INSTRUCTIONS = f"""
Provide a solution(s) to the following university-level engineering exam problem(s):

{refined_question}
"""

In [20]:
# Feed the refined question to the answerer to give an unrefined solution
unrefined_answer_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role" : "system", "content" : UNREFINED_ANSWER_SYSPROMPT},
        {"role" : "user", "content" : UNREFINED_ANSWER_INSTRUCTIONS}
    ]
)
unrefined_answer = unrefined_answer_response.choices[0].message.content

In [21]:
print(unrefined_answer)

To solve this problem, we can break it down into several steps:

1. Analyze the cycle and processes
2. Determine key parameters at each stage of the cycle
3. Calculate the thermal efficiency of the cycle
4. Calculate the specific work output of the turbines per unit mass of steam flowing
5. Calculate the heat transfer for the two stages of heat addition
6. Determine the mass flow rate of steam

Given:
- State 1: P1 = 8 MPa, T1 = 500°C
- State 2: P2 = 2 MPa, T2_reheat = 450°C
- State 3: P3 = 20 kPa
- Isentropic efficiency of turbines, η_t = 90%

1. Analyze the cycle and processes:
- The Rankine cycle with superheat and reheat consists of the following processes:
  a) 1-2: Isentropic compression in the pump
  b) 2-3: Isentropic expansion in the first turbine, reheating to T2_reheat
  c) 3-4: Isentropic expansion in the second turbine

2. Determine key parameters at each stage of the cycle:
State 1 (saturated liquid at 8 MPa):
- h1 = hf = 761.68 kJ/kg (from steam tables)
State 2 (superhea